# 🧬 ESM2 Tools Example

This notebook demonstrates how to use **ESM2** for protein embeddings, mutation sampling, and sequence scoring.

## 📖 What is ESM2?

[ESM2](https://www.science.org/doi/10.1126/science.ade2574) is a state-of-the-art protein language model trained on millions of protein sequences using masked language modeling.

### Key Features:
- 🧠 **Embeddings** - Mean-pooled vector representations for similarity search, clustering, and downstream ML tasks
- 🧪 **Sampling** - Generate mutated sequence variants using model-guided proposals
- 📏 **Scoring** - Pseudo-perplexity via masked scoring to assess sequence naturalness
- ⚙️ **Flexible models** - Multiple model sizes available, from 8M to 15B parameters

## 📦 Imports


## Input/Output Schema

### `ESM2EmbeddingsInput` (alias: `MaskedModelInput`)
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Protein sequence(s) to process as string or list of strings |

### `ESM2EmbeddingsConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| model_checkpoint | str | "esm2_t33_650M_UR50D" | Name of the ESM2 model variant to use |
| batch_size | int | 128 | Number of sequences to process in each batch |
| return_logits | bool | False | Whether to include per-position logits in the output |

### `ESM2EmbeddingsOutput`
| Field | Type | Description |
|-------|------|-------------|
| mean_embeddings | Optional[List[List[float]]] | Mean-pooled sequence embeddings (num_sequences, embedding_dim) |
| num_sequences | int | Number of sequences processed |
| logits | Optional[List[List[List[float]]]] | Per-position amino acid logits (num_sequences, seq_len, 20) |
| attention_masks | List[List[int]] | Attention masks for each sequence (num_sequences, seq_len) |

### `ESM2SampleInput` (alias: `MaskedModelInput`)
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Protein sequence(s) to process as string or list of strings |

### `ESM2SampleConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| model_checkpoint | str | "esm2_t33_650M_UR50D" | Name of the ESM2 model variant to use |
| temperature | float | 1.0 | Sampling temperature for amino acid selection |
| decoding_method | str | "entropy" | Method for selecting positions to mutate ("entropy", "max_logit", "random") |
| num_mutations | int | 1 | Number of positions to mutate per sequence |
| batch_size | Optional[int] | None | Number of sequences to process per batch |
| return_logits | bool | False | Whether to include per-position logits in the output |

### `ESM2SampleOutput`
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Sampled/mutated protein sequences |
| logits | Optional[List[List[List[float]]]] | Per-position amino acid logits (num_sequences, seq_len, 20) |

### `ESM2ScoringInput` (alias: `MaskedModelInput`)
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Protein sequence(s) to process as string or list of strings |

### `ESM2ScoringConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| model_checkpoint | str | "esm2_t33_650M_UR50D" | ESM2 model checkpoint to use |
| batch_size | int | 32 | Number of masked sequences to process per forward pass |
| return_logits | bool | False | Whether to include per-position logits in the output |

### `ESM2ScoringOutput` (alias: `MaskedModelScoringOutput`)
| Field | Type | Description |
|-------|------|-------------|
| scores | List[SequenceScores] | List of scoring outputs, one per input sequence |

### `SequenceScores`
| Field | Type | Description |
|-------|------|-------------|
| metrics | Dict[str, float] | Dictionary of scalar scoring metrics (log_likelihood, avg_log_likelihood, perplexity) |
| logits | Optional[List[List[float]]] | Per-position logits array (seq_len, vocab_size) |
| vocab | Optional[List[str]] | Token ordering for logits |

In [ ]:
from bio_programming_tools.tools.masked_models.esm2 import (
    ESM2EmbeddingsInput, ESM2EmbeddingsConfig, run_esm2_embeddings,
    ESM2SampleInput, ESM2SampleConfig, run_esm2_sample,
    ESM2ScoringInput, ESM2ScoringConfig, run_esm2_score,
)

## 🔍 1. Extract Embeddings

Compute mean-pooled embeddings for protein sequences.

ESM2 generates rich vector representations that capture structural and functional properties of proteins. These embeddings can be used for downstream tasks like clustering, classification, and similarity search.

### Configuration options:
- 🧠 **`model_checkpoint`** - ESM2 model variant (e.g., `esm2_t33_650M_UR50D` for the 650M parameter model)
- 📦 **`batch_size`** - Number of sequences to process in parallel
- 📊 **`return_logits`** - Whether to return per-position amino acid log-probabilities


In [2]:
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM2EmbeddingsInput(sequences=sequences)

# Configure the embedding model
config = ESM2EmbeddingsConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    batch_size=2,
    return_logits=False,
)

embeddings_result = run_esm2_embeddings(inputs, config)
print(f"Processed {embeddings_result.num_sequences} sequences")
print(f"Embedding dim: {len(embeddings_result.mean_embeddings[0])}")


Processed 2 sequences
Embedding dim: 1280


## 🧪 2. Sample Mutations

Generate mutated sequence variants using model-guided proposals.

ESM2 uses its learned amino acid distributions to propose mutations at positions where the model is most uncertain (entropy-based) or most confident, depending on the decoding strategy.

### Configuration options:
- 🧠 **`model_checkpoint`** - ESM2 model variant (e.g., `esm2_t33_650M_UR50D` for the 650M parameter model)
- 🎲 **`decoding_method`** - Strategy for selecting which positions to mutate
  - `"entropy"` - Mutate positions with highest prediction uncertainty (default)
  - `"max_logit"` - Mutate positions with lowest confidence predictions
  - `"random"` - Randomly select positions to mutate
- 🌡️ **`temperature`** - Controls randomness in amino acid selection (lower = more conservative, higher = more diverse)
- 🔢 **`num_mutations`** - Number of positions to mutate per sequence


In [3]:
inputs = ESM2SampleInput(sequences=["MVLSPADKTNVKAAW"])

# Configure mutation sampling
config = ESM2SampleConfig(
    decoding_method="entropy",
    num_mutations=5,
)

sample_result = run_esm2_sample(inputs, config)
print(f"Original: {inputs.sequences[0]}")
print(f"Mutated:  {sample_result.sequences[0]}")


Original: MVLSPADKTNVKAAW
Mutated:  MVLSSADGTNVKAAK


## 📏 3. Score Sequences

Compute pseudo-perplexity metrics for each sequence.

Pseudo-perplexity measures how "natural" a protein sequence looks to the model. Lower values indicate sequences that are more consistent with the patterns learned from natural proteins. This is useful for ranking designed sequences or identifying anomalous regions.

### Configuration options:
- 🧠 **`model_checkpoint`** - ESM2 model variant (e.g., `esm2_t33_650M_UR50D` for the 650M parameter model)
- 📦 **`batch_size`** - Number of masked variants to process per forward pass
- 📊 **`return_logits`** - Whether to return per-position amino acid log-probabilities


In [4]:
inputs = ESM2ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

# Configure scoring parameters
config = ESM2ScoringConfig(batch_size=5, return_logits=False)

score_result = run_esm2_score(inputs, config)
print(f"Sequence 1: {inputs.sequences[0]}")
print(f"    Perplexity: {score_result.scores[0].metrics['perplexity']:.3f}")
print(f"Sequence 2: {inputs.sequences[1]}")
print(f"    Perplexity: {score_result.scores[1].metrics['perplexity']:.3f}")


Sequence 1: MKTAYIAKQR
    Perplexity: 15.246
Sequence 2: EVQLVESGGS
    Perplexity: 17.110


## 💾 4. Export Results

Save the results to files for downstream analysis.

### Supported formats:
- 📄 **JSON** - Structured data with metadata and all scoring information
- 📊 **CSV** - Tabular format for embeddings, compatible with pandas and other data tools
- 🧬 **FASTA** - Simple sequence format for compatibility with other bioinformatics tools

The exported results can be used for:
- Downstream machine learning (embeddings as features)
- Sequence analysis and comparison (sampled variants)
- Ranking and filtering protein designs (scoring metrics)

In [ ]:
# Export embeddings to CSV format
embeddings_result.export("./example_output/esm2_embeddings", file_format="csv")
print("Embeddings exported to ./example_output/esm2_embeddings")

# Export sampled sequences to FASTA format
sample_result.export("./example_output/esm2_sequences", file_format="fasta")
print("Sampled sequences exported to ./example_output/esm2_sequences")

# Export scores to JSON format
score_result.export("./example_output/esm2_scores", file_format="json")
print("Scores exported to ./example_output/esm2_scores")